# Underfitting și overfitting

Material ONIA — Resurse educaționale. Exercițiile pot fi modificate și executate în JupyterLab.

## Ideea centrală: a învăța nu înseamnă a memora

Un elev poate memora răspunsurile la zece exerciții fără să știe să rezolve un al unsprezecelea. Similar, un model poate învăța și coincidențe din exemplele de antrenare. Generalizarea este capacitatea de a funcționa bine pe date noi, reprezentative pentru aceeași problemă. Separăm eroarea pe train de eroarea pe validation. Dacă ambele sunt mari, modelul poate fi prea simplu (underfitting); dacă train este excelent și validation slabă, avem un semnal de overfitting. Acestea sunt indicii, nu diagnostice automate: un set de validare nepotrivit poate induce în eroare.

## Underfitting — modelul nu surprinde tiparul

Într-o problemă în care dependența dintre x și y este curbată, o singură dreaptă poate să nu fie suficientă. Modelul face greșeli mari atât la punctele deja văzute, cât și la cele noi. Cauze: model prea restrictiv, caracteristici insuficiente, regularizare excesivă, antrenare insuficientă sau o reprezentare defectuoasă a intrărilor. Remedii posibile: caracteristici relevante, model mai flexibil, scăderea regularizării ori optimizarea antrenării — verificând fiecare modificare pe validare.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
X = [[-2], [-1], [0], [1], [2]]
y = [4, 1, 0, 1, 4]
dreapta = LinearRegression().fit(X, y)
parabola = make_pipeline(PolynomialFeatures(2), LinearRegression())
parabola.fit(X, y)
print('La x=3:', dreapta.predict([[3]])[0], parabola.predict([[3]])[0])

## Overfitting — când modelul învață zgomotul

Dacă ajustăm un polinom de grad foarte mare pentru puține puncte, el poate trece aproape prin fiecare observație, inclusiv prin zgomot. Între puncte poate oscila violent. Eroarea pe train scade, dar aceea pe validation poate crește. În clasificare, un arbore fără limite de adâncime poate crea câte o frunză pentru fiecare observație. În K-NN, k=1 poate deveni sensibil la erori de etichetare. Nu înseamnă că aceste setări sunt mereu greșite: depinde de date și de validarea corectă.

## Bias și variance pe înțelesul liceenilor

Bias reprezintă eroarea sistematică introdusă de ipoteze prea restrictive: o dreaptă nu poate aproxima bine orice curbă. Variance este sensibilitatea soluției la schimbări mici ale datelor: un model foarte flexibil își poate schimba dramatic forma când înlocuim câteva exemple. O simplificare utilă: underfitting este frecvent asociat cu bias ridicat; overfitting, cu variance ridicată. În realitate, cele două pot coexista și depind de algoritm, distribuție, zgomot și procedura de antrenare.

## Curbe de învățare: câte exemple ne mai trebuie?

Reprezentăm eroarea pe antrenare și validare în funcție de numărul de exemple folosite. Dacă există un decalaj mare între train și validare, mai multe exemple pot ajuta în anumite condiții. Dacă ambele erori sunt mari și apropiate, probabil că nu este suficient să colectăm mai multe observații: trebuie îmbunătățite modelul sau caracteristicile. Compară mereu scorurile în aceeași metrică, pe seturi comparabile și cu aleatorie controlată.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import learning_curve
from sklearn.linear_model import Ridge
X, y = make_regression(n_samples=150, n_features=5, noise=15, random_state=42)
sizes, train_scores, val_scores = learning_curve(
    Ridge(alpha=1), X, y,
    train_sizes=[0.3, 0.6, 1.0], cv=5, scoring='neg_mean_squared_error'
)
print(sizes)
print((-val_scores.mean(axis=1)).round(1))

## Măsuri împotriva overfittingului

Acțiunile uzuale sunt: folosirea unui set de validare separat, regularizare L1/L2, limitarea adâncimii arborilor, reducerea complexității, colectarea de observații reprezentative, eliminarea caracteristicilor redundante și early stopping (când este aplicabil). În modelele neurale, dropout și augmentarea de date pot avea un rol. Nu este corect să presupui că o regularizare mai puternică este automat mai bună. Dacă este prea puternică, modelul poate ajunge la underfitting.

## Public leaderboard și private leaderboard

Într-o competiție, fișierul trimis este evaluat pe date ascunse, iar rezultatul public poate fi calculat pe un subset. Scorul final poate fi obținut pe alte observații, nevăzute până la sfârșitul competiției. Repetarea încercărilor exclusiv pentru a îmbunătăți câteva miimi pe scorul public poate duce la leaderboard overfitting . Exemplu fictiv: modelul A are scor public 0,92 și final 0,82, iar modelul B are public 0,90 și final 0,89. În acest exemplu, B generalizează mai bine pe subsetul final. Nu este o regulă că ordinea se inversează mereu, ci un avertisment că scorul public este o estimare parțială.

## Data leakage și separări incorecte

Dacă aceleași imagini aproape identice apar atât în train, cât și în validation, performanța estimată va fi artificial de mare. La date temporale, amestecarea observațiilor din viitor cu cele din trecut poate crea informație indisponibilă la predicție. La pacienți, imagini ale aceleiași persoane trebuie deseori ținute în același grup. Înainte de a crește complexitatea modelului, verifică sursele posibile de scurgere de informație.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
X = [[i] for i in range(12)]
y = [i % 2 for i in range(12)]
grupe = [i // 3 for i in range(12)]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=grupe))
print('Grupuri train:', {grupe[i] for i in train_idx})
print('Grupuri test:', {grupe[i] for i in test_idx})

## Cross-validation și setul final

În K-fold cross-validation, împărțim datele de dezvoltare în K bucăți; de K ori antrenăm pe K−1 bucăți și evaluăm pe cea rămasă. Media și variația scorurilor sunt mai informative decât un singur split. Pentru date temporale se recomandă un split care respectă timpul, iar pentru grupuri corelate un split după grupuri. După alegerea hiperparametrilor, antrenezi modelul final pe datele de dezvoltare relevante; setul de test final nu trebuie folosit pentru ajustarea hiperparametrilor.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
X, y = load_iris(return_X_y=True)
scoruri = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5)
print('Scoruri pe folduri:', scoruri.round(3))
print('Media:', round(scoruri.mean(), 3))

## Checklist înainte de trimiterea la evaluare

Verifică dacă fișierul trimis are formatul cerut, ordinea observațiilor este păstrată, probabilitățile sunt între 0 și 1 când problema cere probabilități, iar vectorii de predicție au lungimea corectă. O valoare de random_state ajută la reproductibilitate, dar nu înlocuiește o validare corectă. Documentează experimentul ales: caracteristici, hiperparametri, seed, metrică pe validation și observații despre erori.

In [ ]:
import numpy as np
probabilitati = np.array([0.1, 0.83, 0.49])
assert probabilitati.ndim == 1
assert np.isfinite(probabilitati).all()
assert np.all((0 <= probabilitati) & (probabilitati <= 1))
print('Predicțiile au format valid')